# 19 - Final Ablation Matrix and Claims Register

## Research question

Which claims survive the full v5 campaign and can safely enter the manuscript rewrite?

## Artifact paths loaded

- `results/v5/final_ablation_matrix.csv`
- `results/v5/final_ablation_matrix.json`
- `results/v5/final_v5_model_comparison.csv`
- `reports/v5_campaign/claims_register_v2.json`
- `reports/v5_campaign/claims_register_v2.md`
- `reports/v5_campaign/phase10_readiness_gate.json`
- `reports/v5_campaign/phase10_manuscript_readiness_gate.md`
- `reports/v5_campaign/phase7_final_ablation_model_card.md`

All cells are analysis-only. No heavy training is run.


In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown, Image
except Exception:
    def display(x): print(x)
    def Markdown(x): return x
    def Image(filename=None, **kwargs): return f"Image({filename})"

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "legend.frameon": False,
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
})
COLORS = ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51", "#0072B2", "#56B4E9", "#8C8C8C"]
OUR_COLOR = "#E76F51"
BASELINE_COLOR = "#B0BEC5"

def find_repo_root():
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "results" / "v5").exists() and (p / "reports" / "v5_campaign").exists():
            return p
    raise RuntimeError("Could not find repository root containing results/v5 and reports/v5_campaign")

ROOT = find_repo_root()
RESULTS = ROOT / "results" / "v5"
REPORTS = ROOT / "reports" / "v5_campaign"
FIGS = RESULTS / "figures"
missing_artifacts = []

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT)).replace("\\", "/")
    except Exception:
        return str(path).replace("\\", "/")

def artifact(path):
    p = ROOT / path if isinstance(path, str) else Path(path)
    if not p.exists():
        missing_artifacts.append(rel(p))
    return p

def read_csv_safe(path):
    p = artifact(path)
    if not p.exists():
        display(Markdown(f"**Missing artifact:** `{rel(p)}`"))
        return pd.DataFrame()
    return pd.read_csv(p)

def read_json_safe(path):
    p = artifact(path)
    if not p.exists():
        display(Markdown(f"**Missing artifact:** `{rel(p)}`"))
        return {}
    with p.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def show_artifact_status(paths):
    rows = []
    for item in paths:
        p = ROOT / item
        rows.append({"artifact": item, "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else None})
        if not p.exists():
            missing_artifacts.append(item)
    display(pd.DataFrame(rows))

def show_missing():
    unique = sorted(set(missing_artifacts))
    if unique:
        display(Markdown("### Missing artifacts recorded by this notebook"))
        display(pd.DataFrame({"missing_artifact": unique}))
    else:
        display(Markdown("### Missing artifacts recorded by this notebook: none"))

def maybe_display_png(path):
    p = ROOT / path if isinstance(path, str) else Path(path)
    if p.exists():
        display(Image(filename=str(p)))
    else:
        missing_artifacts.append(rel(p))
        display(Markdown(f"Existing figure not found: `{rel(p)}`"))

print("Repository root:", ROOT)


Repository root: D:\Tugas kuliah\SEM 6\PROYEK DATA MINING\PENELITIAN\Penelitian_SOC


In [2]:
ARTIFACTS = [
    'results/v5/final_ablation_matrix.csv',
    'results/v5/final_ablation_matrix.json',
    'results/v5/final_v5_model_comparison.csv',
    'reports/v5_campaign/claims_register_v2.json',
    'reports/v5_campaign/claims_register_v2.md',
    'reports/v5_campaign/phase10_readiness_gate.json',
    'reports/v5_campaign/phase10_manuscript_readiness_gate.md',
    'reports/v5_campaign/phase7_final_ablation_model_card.md',
]
show_artifact_status(ARTIFACTS)

matrix = read_csv_safe("results/v5/final_ablation_matrix.csv")
final = read_csv_safe("results/v5/final_v5_model_comparison.csv")
claims = read_json_safe("reports/v5_campaign/claims_register_v2.json")
gate = read_json_safe("reports/v5_campaign/phase10_readiness_gate.json")


,artifact,exists,bytes
0,results/v5/final_ablation_matrix.csv,True,11046
1,results/v5/final_ablation_matrix.json,True,37467
2,results/v5/final_v5_model_comparison.csv,True,5653
3,reports/v5_campaign/claims_register_v2.json,True,6152
4,reports/v5_campaign/claims_register_v2.md,True,5806
5,reports/v5_campaign/phase10_readiness_gate.json,True,5768
6,reports/v5_campaign/phase10_manuscript_readine...,True,1740
7,reports/v5_campaign/phase7_final_ablation_mode...,True,3962


In [3]:
if not matrix.empty:
    display(Markdown(f"### Final ablation matrix: {len(matrix)} rows"))
    display(matrix.groupby(["dataset_version","source"]).size().reset_index(name="rows").sort_values(["dataset_version","source"]))
    display(matrix.head(12).round(4))


### Final ablation matrix: 62 rows

,dataset_version,source,rows
0,v4_legacy,legacy_freeze_manifest.json,12
1,v5c,baselines/deterministic_baselines_v5c.json,8
2,v5c,delta_calibration/eta_gamma_sweep.json,16
3,v5c,ekf_ecm/continuous_ekf_results.json,9
4,v5c,multiseed/seed_level_results.csv,10
5,v5c,recursive_inference/recursive_policy_results.json,7


,dataset_version,label_mode,decimation_mode,scenario,model,anchor_strategy,inference_policy,eta,gamma_mode,seeds,baseline_type,rmse_pct,rmse_std,maxe_pct,rmse_n20,rmse_40,pvr_disch_eps0,source
0,v4_legacy,legacy,first_sample,scenario_A,vanilla_lstm,NaN,windowed,1.5,nominal,42(v4),frozen_v4,13.3712,NaN,51.0242,NaN,NaN,49.9694,legacy_freeze_manifest.json
1,v4_legacy,legacy,first_sample,scenario_A,hard_coulomb_lstm,NaN,windowed,1.5,nominal,42(v4),frozen_v4,12.7107,NaN,55.1126,17.8587,NaN,0.0000,legacy_freeze_manifest.json
2,v4_legacy,legacy,first_sample,scenario_A,hard_coulomb_tcn,NaN,windowed,1.5,nominal,42(v4),frozen_v4,11.4592,NaN,46.7303,NaN,NaN,NaN,legacy_freeze_manifest.json
3,v4_legacy,legacy,first_sample,scenario_A,null_ocv_coulomb_ocv25_qnom,NaN,windowed,1.5,nominal,42(v4),frozen_v4,13.2436,NaN,86.9666,NaN,NaN,NaN,legacy_freeze_manifest.json
4,v4_legacy,legacy,first_sample,scenario_A,vanilla_posthoc_clamp,NaN,windowed,1.5,nominal,42(v4),frozen_v4,25.5561,NaN,NaN,NaN,NaN,NaN,legacy_freeze_manifest.json
5,v4_legacy,legacy,first_sample,scenario_A,hc_lstm_recursive_infer,NaN,windowed,1.5,nominal,42(v4),frozen_v4,9.2251,NaN,31.1895,6.6815,NaN,NaN,legacy_freeze_manifest.json
6,v4_legacy,legacy,first_sample,scenario_A,hc_lstm_oracle_anchor_ref,NaN,windowed,1.5,nominal,42(v4),frozen_v4,0.5379,NaN,5.0462,0.4964,NaN,NaN,legacy_freeze_manifest.json
7,v4_legacy,legacy,first_sample,scenario_A,ekf_ocv_rint_best,NaN,windowed,1.5,nominal,42(v4),frozen_v4,13.0002,NaN,99.0480,NaN,NaN,30.2887,legacy_freeze_manifest.json
8,v4_legacy,legacy,first_sample,scenario_B,vanilla_lstm,NaN,windowed,1.5,nominal,42(v4),frozen_v4,7.2806,NaN,NaN,NaN,NaN,NaN,legacy_freeze_manifest.json
9,v4_legacy,legacy,first_sample,scenario_B,hard_coulomb_lstm,NaN,windowed,1.5,nominal,42(v4),frozen_v4,8.5667,NaN,34.9985,NaN,NaN,NaN,legacy_freeze_manifest.json


In [4]:
if not final.empty:
    selected = final[(final["model"]=="hc_anchor_last") | (final["model"].astype(str).str.contains("HC_|EKF|hard_coulomb_lstm|vanilla_lstm|null", regex=True, na=False))]
    display(Markdown("### Final model comparison excerpt"))
    display(selected[["scenario","model","anchor_strategy","inference_policy","eta","gamma_mode","seeds","baseline_type","rmse_pct","rmse_std","maxe_pct","rmse_n20","pvr_disch_eps0","source"]].round(4))
    display(Markdown("**Selected final system:** `anchor_last + calibrated carried inference`. Windowed anchor_last is the trained model selection; eta-calibrated carried inference is the final recursive deployment policy."))


### Final model comparison excerpt

,scenario,model,anchor_strategy,inference_policy,eta,gamma_mode,seeds,baseline_type,rmse_pct,rmse_std,maxe_pct,rmse_n20,pvr_disch_eps0,source
0,scenario_A,hard_coulomb_lstm,first,windowed,1.5,nominal,1-5,NaN,10.8740,0.1904,46.4673,16.3274,0.0000,multiseed/seed_level_results.csv
2,scenario_A,hc_anchor_last,last,windowed,1.5,nominal,1-5,NaN,9.9930,1.0897,36.4976,15.0454,0.0000,multiseed/seed_level_results.csv
4,scenario_A,vanilla_lstm,NaN,windowed,1.5,nominal,1-5,NaN,11.3474,0.1798,47.4796,17.2103,45.8981,multiseed/seed_level_results.csv
5,scenario_B,hard_coulomb_lstm,first,windowed,1.5,nominal,1-5,NaN,10.6291,0.5982,35.5544,7.7697,0.0000,multiseed/seed_level_results.csv
7,scenario_B,hc_anchor_last,last,windowed,1.5,nominal,1-5,NaN,4.7432,0.3141,34.4872,8.0530,0.0000,multiseed/seed_level_results.csv
9,scenario_B,vanilla_lstm,NaN,windowed,1.5,nominal,1-5,NaN,6.1997,0.2718,50.8375,8.6908,37.2987,multiseed/seed_level_results.csv
10,scenario_A,null[ocv25_qnom],NaN,windowed,1.5,nominal,NaN,deterministic,13.0297,NaN,60.9376,19.5747,0.0000,baselines/deterministic_baselines_v5c.json
11,scenario_A,null[ocv25_qtemp],NaN,windowed,1.5,nominal,NaN,deterministic,13.1430,NaN,60.1359,19.7392,0.0000,baselines/deterministic_baselines_v5c.json
12,scenario_A,null[oracle_qtemp],NaN,windowed,1.5,nominal,NaN,deterministic,0.0257,NaN,0.2334,0.0263,0.0000,baselines/deterministic_baselines_v5c.json
14,scenario_B,null[ocv25_qnom],NaN,windowed,1.5,nominal,NaN,deterministic,7.5278,NaN,94.7159,14.3190,0.0000,baselines/deterministic_baselines_v5c.json


**Selected final system:** `anchor_last + calibrated carried inference`. Windowed anchor_last is the trained model selection; eta-calibrated carried inference is the final recursive deployment policy.

In [5]:
claim_rows = []
for c in claims.get("claims", []) if isinstance(claims, dict) else []:
    claim_rows.append({"id": c.get("id"), "status": c.get("status"), "claim": c.get("claim"), "evidence": c.get("evidence"), "sources": ", ".join(c.get("sources", [])) if isinstance(c.get("sources"), list) else c.get("sources")})
claim_df = pd.DataFrame(claim_rows)
if not claim_df.empty:
    display(Markdown(f"### Claims register v{claims.get('version', '?')}: {len(claim_df)} claims"))
    display(claim_df)
    display(claim_df["status"].value_counts().rename_axis("status").reset_index(name="count"))


### Claims register v2: 17 claims

,id,status,claim,evidence,sources
0,1,SUPPORTED,HC enforces sign-consistency w.r.t. measured c...,"PVR==0 across all v5c seeds, policies, eta val...","P2, P3, P4, P5, P6"
1,2,SUPPORTED,"PVR 0.00% stated 'by construction', not as result",every v5 table footnotes it,P2
2,3,SUPPORTED_FOR_ANCHOR_VARIANTS,HC improves RMSE over baselines (general),anchor_last/pooled beat vanilla AND null both ...,"P2, P3"
3,4,SUPPORTED_ANCHOR_VARIANTS_PARTIAL_ORIGINAL,HC beats null OCV+Coulomb,anchor_last: everywhere every seed; original H...,"P2, P3"
4,5,SUPPORTED,HC beats post-hoc clamp,clamp collapse on v5c: 25.0/30.1% vs HC <=10.9...,P2
5,6,SUPPORTED,-20C error is anchor-dominated,oracle-anchor null 0.03% vs learned 15-17%; in...,"P2, P4, P5"
6,7,UNSUPPORTED,Windowed evaluation reflects deployment,protocol-dependent; frontier quantified: calib...,"P4, P5"
7,8,LARGELY_RESOLVED_BY_V5C,Labels are trustworthy,v5c corrects ohmic-anchor bias + decimation co...,"P1, P2"
8,9,PARTIALLY_SUPPORTED,Edge deployment feasible (parameter-level),54.6k params / 218KB / MCU-class MMAC; no hard...,v4_P9
9,10,PARTIALLY_SUPPORTED,PVR survives quantization (path-dependent),single-scale-accumulator requirement stands,v4_P9


,status,count
0,SUPPORTED,8
1,UNSUPPORTED,2
2,PARTIALLY_SUPPORTED,2
3,SUPPORTED_FOR_ANCHOR_VARIANTS,1
4,SUPPORTED_ANCHOR_VARIANTS_PARTIAL_ORIGINAL,1
5,LARGELY_RESOLVED_BY_V5C,1
6,UNSUPPORTED_AS_ROBUSTNESS_SUPPORTED_AS_CHARACT...,1
7,SUPPORTED_SINGLE_SEED,1


In [6]:
if isinstance(gate, dict) and gate:
    display(Markdown(f"### Manuscript readiness gate: {gate.get('passed')}/{gate.get('total')} PASS"))
    gates = pd.DataFrame(gate.get("gates", []))
    display(gates.head(20))
    if "pass" in gates.columns:
        display(gates["pass"].value_counts().rename_axis("pass").reset_index(name="count"))


### Manuscript readiness gate: 52/52 PASS

,name,pass,detail
0,artifact:legacy_freeze_manifest.json,True,
1,artifact:dataset_variant_comparison.json,True,
2,artifact:headline_models/runs_v5c_A-B_42.json,True,
3,artifact:headline_models/v4_vs_v5_comparison.csv,True,
4,artifact:multiseed/runs_A.json,True,
5,artifact:multiseed/runs_B.json,True,
6,artifact:multiseed/multiseed_summary.csv,True,
7,artifact:multiseed/ranking_stability.json,True,
8,artifact:recursive_inference/recursive_policy_...,True,
9,artifact:delta_calibration/eta_gamma_sweep.json,True,


,pass,count
0,True,52


In [7]:
if not claim_df.empty:
    safe = claim_df[claim_df["status"].astype(str).str.contains("SUPPORTED", na=False)]
    dead = claim_df[claim_df["status"].astype(str).str.contains("UNSUPPORTED", na=False)]
    display(Markdown("### Safe claims for manuscript"))
    display(safe[["id","status","claim","evidence"]])
    display(Markdown("### Dead or restricted claims"))
    display(dead[["id","status","claim","evidence"]])


### Safe claims for manuscript

,id,status,claim,evidence
0,1,SUPPORTED,HC enforces sign-consistency w.r.t. measured c...,"PVR==0 across all v5c seeds, policies, eta val..."
1,2,SUPPORTED,"PVR 0.00% stated 'by construction', not as result",every v5 table footnotes it
2,3,SUPPORTED_FOR_ANCHOR_VARIANTS,HC improves RMSE over baselines (general),anchor_last/pooled beat vanilla AND null both ...
3,4,SUPPORTED_ANCHOR_VARIANTS_PARTIAL_ORIGINAL,HC beats null OCV+Coulomb,anchor_last: everywhere every seed; original H...
4,5,SUPPORTED,HC beats post-hoc clamp,clamp collapse on v5c: 25.0/30.1% vs HC <=10.9...
5,6,SUPPORTED,-20C error is anchor-dominated,oracle-anchor null 0.03% vs learned 15-17%; in...
6,7,UNSUPPORTED,Windowed evaluation reflects deployment,protocol-dependent; frontier quantified: calib...
8,9,PARTIALLY_SUPPORTED,Edge deployment feasible (parameter-level),54.6k params / 218KB / MCU-class MMAC; no hard...
9,10,PARTIALLY_SUPPORTED,PVR survives quantization (path-dependent),single-scale-accumulator requirement stands
10,11,UNSUPPORTED,Functional safety supported,only 'safety-motivated architectural property'...


### Dead or restricted claims

,id,status,claim,evidence
6,7,UNSUPPORTED,Windowed evaluation reflects deployment,protocol-dependent; frontier quantified: calib...
10,11,UNSUPPORTED,Functional safety supported,only 'safety-motivated architectural property'...
11,12,UNSUPPORTED_AS_ROBUSTNESS_SUPPORTED_AS_CHARACT...,Sensor-fault robustness,characterized failure envelope


## Interpretation

The final v5 paper should be written from the ablation matrix and claims register, not from isolated old notebooks. The selected system is conservative: anchor_last for stable windowed model selection, calibrated carried inference for recursive operation, and load-gated re-anchoring when chain continuity is unreliable.

## Reviewer-risk note

Do not revive unsupported claims: no full functional-safety compliance, no hardware readiness without WCET/INT8/HIL, no universal eta constant without additional validation.

## Final conclusion

Ready for manuscript rewrite with conservative claims.


In [8]:
show_missing()


### Missing artifacts recorded by this notebook: none

## Publication asset export (PUBLICATION_ASSET_EXPORT_V1)

This cell exports manuscript-ready figures/tables from existing v5 CSV/JSON artifacts only. It does not run training or alter experiment results.


In [ ]:
# PUBLICATION_ASSET_EXPORT_V1
from pathlib import Path
import sys
EXPORT_DIR = Path('notebooks/ablation_studies_v5_final').resolve()
if str(EXPORT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPORT_DIR))
from publication_asset_exports import export_for_notebook
validation = export_for_notebook('19')
print('publication assets refreshed:', validation['png_present'], 'figures and', validation['csv_present'], 'tables')
